In [1]:
import sys, os
sys.path.insert(0, 'src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from data_loader import load_raw_data, get_dataset_metadata
from preprocessing import preprocess_pipeline
raw_df = load_raw_data()
metadata = get_dataset_metadata(raw_df)
print('Raw Dataset Metadata:', metadata)
clean_df, sensor_report = preprocess_pipeline(raw_df)
print(f'Preprocessed Dataset: {len(clean_df):,} rows x {clean_df.shape[1]} columns')
print('Sensor Anomaly Treatment Report:', sensor_report)


In [3]:
from eda import run_full_eda
eda_results = run_full_eda(clean_df, output_dir='outputs/figures')
print('Statistical Profiling Summary:')
print(eda_results['summary_table'].head(10))


In [4]:
from anomaly_detection import run_full_anomaly_detection
anom_df, anom_results = run_full_anomaly_detection(clean_df, fig_dir='outputs/figures', metric_dir='outputs/metrics')
print('Detected Weather Anomalies Summary:')
for metric_name, val in anom_results['metrics'].items():
    print(f'  - {metric_name}: {val}')


In [5]:
from models import run_full_forecasting_pipeline
from evaluation import plot_model_comparison, plot_residuals, plot_actual_vs_predicted
metrics_table, predictions, model_artifacts = run_full_forecasting_pipeline(clean_df)
plot_model_comparison(metrics_table, output_dir='outputs/figures')
y_test = model_artifacts['y_test']
res_dict = {name: (y_test, preds) for name, preds in predictions.items() if 'Baseline' not in name}
plot_residuals(res_dict, output_dir='outputs/figures')
best_model = metrics_table.iloc[0]['model_name']
plot_actual_vs_predicted(y_test, predictions[best_model], model_name=best_model, output_dir='outputs/figures')
print('Model Evaluation Benchmark:')
print(metrics_table.to_string(index=False))


In [6]:
from advanced_analyses import run_all_advanced_analyses
test_df = model_artifacts['test_df']
feat_cols = model_artifacts['feature_cols']
rf_model = model_artifacts['models']['RandomForest']
X_test = test_df[feat_cols].values
adv_results = run_all_advanced_analyses(
    clean_df, rf_model, feat_cols, X_test, y_test,
    fig_dir='outputs/figures', metric_dir='outputs/metrics'
)
print('All 5 Advanced Unique Analyses completed successfully!')
